<img src='https://github.com/wekeo/ai4EM_MOOC/raw/04147f290cfdcce341f819eab7ad037b95f25600/img/ai4eo_logos.jpg' alt='Logo EU Copernicus WEkEO' align='center' width='100%'></img>


# WEKEO TRAINING

<div style="text-align: right"><i> INTERMEDIATE LEVEL </i></div>

***
<center><h1> Etna Eruption in June 2025 </h1></center>

***
**General Note 1**: Execute each cell through the <button class="btn btn-default btn-xs"><i class="icon-play fa fa-play"></i></button> button from the top MENU (or keyboard shortcut `Shift` + `Enter`).<br>
<br>
**General Note 2**: If, for any reason, the kernel is not working anymore, in the top MENU, click on the <button class="btn btn-default btn-xs"><i class="fa fa-repeat icon-repeat"></i></button> button. Then, in the top MENU, click on "Run" and select "Run All Above Selected Cell".<br>

**General Note 3**: To explore more (Python and R) content, there is our [**Jupyter Catalogue**](https://notebooks.apps.mercator.dpi.wekeo.eu/), and if you seek support, there are plenty of useful resources in our [**Help Center**](https://help.wekeo.eu/en/). Feel free to contact us using our [**live chat widget**](https://www.wekeo.eu/support) or asynchronously at [support@wekeo.eu](support@wekeo.eu) ! <br>

***

# Table of contents
- [1. Introduction](#1.-Introduction)
- [2. Setting up the R environment](#2.-Setting-up-the-R-environment)
- [3. Data Access](#3.-Data-Access)
- [4. Input for Sentinel 5P](#4.-Input-for-Sentinel-5P)
- [5. Sentinel 5P image](#5.-Sentinel-5P-image)
    - [5.1. Plot of sulfur dioxide](#5.1.-Plot-of-sulfur-dioxide)
    - [5.2. Sulfur dioxide map with country borders](#5.2.-Sulfur-dioxide-map-with-country-borders)
- [6. Use of Sentinel 2 data](#6.-Use-of-Sentinel-2-data)
    - [6.1 RGB Image During the Eruption](#6.1-RGB-Image-During-the-Eruption)
    - [6.2 Calculation of NBR during eruption](#6.2-Calculation-of-NBR-during-eruption)
    - [6.3. Comparison between during and after the eruption](#6.3.-Comparison-between-during-and-after-the-eruption)
        - [6.3.1. RGB images](#6.3.1.-RGB-images)
        - [6.3.2. NBR images](#6.3.2.-NBR-images)
- [7. Visualisation of wind ](#7.-Visualisation-of-wind)
- [8. Conclusion](#8.-Conclusion)


# 1. Introduction

[Go back to the "Table of contents"](#Table-of-contents)

This training focuses on the June 2025 eruption of Mount Etna, one of the most active volcanoes in Europe. On June 2, a powerful strombolian explosion from Etna’s Southeast Crater generated vigorous lava flows down the volcano’s flanks and propelled a dense plume of ash, gas and volcanic material several kilometres into the atmosphere, as observed by satellites and Earth observation networks.

In this training, we will visualize the extent of the lava flow and highlight areas affected or unaffected by the lava. In parallel, SO₂ measurements from Sentinel‑5P TROPOMI will allow us to track the volcanic plume and estimate the amount of SO₂ released into the atmosphere during the eruption.
To understand how volcanic ash and gases are transported away from the volcano, we will incorporate ERA5 reanalysis meteorological data, which provide comprehensive information on wind fields and atmospheric conditions that influence the dispersion of ash and SO₂ clouds.

# 2. Setting up the R environment
[Go back to the "Table of contents"](#Table-of-contents)

The Jupyter Notebook must be set up with all the necessary available tools from the Jupyter Notebook ecosystem. Here is the list of the modules we will be using in this exercise.

| Module name | Description |
| :---: | :---|
| **ggplot2** |[ggplot2](https://ggplot2.tidyverse.org/) is a system for declaratively creating graphics. |
| **dplyr** |[dplyr](https://dplyr.tidyverse.org/) is a library for manipulating data. |
| **RColorBrewer** |[RColorBrewer](https://cran.r-project.org/web/packages/RColorBrewer/index.html) is a library for color palettes in plot. |
| **ncdf4** |[ncdf4](https://cran.r-project.org/web/packages/ncdf4/index.html) is an interface for the 'NetCDF' file formats. |
| **gridExtra** |[gridExtra](https://cran.r-project.org/web/packages/gridExtra/index.html) is a library for arranging multiple grid-based plots on a page. |
| **lubridate** |[lubridate](https://www.rdocumentation.org/packages/lubridate/versions/1.9.3) is a library for fast and user-friendly analysis of date-time data. |
|**raster**|[raster](https://www.rdocumentation.org/packages/raster/versions/3.6-30/topics/raster) is a package for reading, writing, manipulating, analyzing and modeling of spatial data. |
|**akima**|[akima](https://www.rdocumentation.org/packages/akima/versions/0.4-3) is a package for interpolating irregularly spaced data onto a regular grid in two dimensions. |
|**rnaturalearth & rnaturalearthdata**|[rnaturalearth](https://cran.r-project.org/web/packages/rnaturalearth/vignettes/rnaturalearth.html) is a data package designed to provide map data . |
|**sf**|[sf](https://cran.r-project.org/web/packages/sf/index.html) is a package for standardized way to encode and analyze spatial vector data. |
| **ggquiver** |[ggquiver](https://cran.r-project.org/web/packages/ggquiver/readme/README.html) is a library for creating quiver plots for ggplot2. |
| **scales** |[scales](https://cran.r-project.org/web/packages/scales/index.html) is a library for determing graphical scales that map data to aesthetics. |

In [ ]:
## Installation of required packages
packages <- c("akima", "rnaturalearth", "rnaturalearthdata", "ggquiver") 

for (pkg in packages) { 
  if (!requireNamespace(pkg, quietly = TRUE)) { 
    cat("Installing", pkg, "...\n") 
    install.packages(pkg, quiet = TRUE) 
    cat(pkg, "installed successfully\n\n") 
  } else { 
    cat(pkg, "is already installed\n\n") 
  } 
} 


In [ ]:
# Modules system
# for ignoring Warning message
options(warn = -1) 

## Load Packages
suppressPackageStartupMessages({ 
    library(ggplot2)        # System to creating graphycs
    library(dplyr)          # Operators for data manipulation
    library(RColorBrewer)   # For color palettes
    library(ncdf4)          # High-level interface to netCDF data files
    library(gridExtra)      # To arrange multiple grid-based plots on a page
    library(lubridate)      # Fast and user friendly parsing of date-time data
    library(raster)         # Toolkit for image processing
    library(akima)          # to interpolate irregularly spaced data onto a regular grid in two dimensions.
    library(rnaturalearth)  # to add background map
    library(rnaturalearthdata) # contain data
    library(sf)            # to manipulate vector data 
    library(terra)         # for handling raster and vector spatial data
    library(ggquiver)      # To visualise vector fields
    library(scales)        # Scales map data to aesthetics
})


If the required R libraries are not installed, you can use the following command to install them:

<center><h4> install.packages(c("ggplot2", "dplyr", "RColorBrewer", "ncdf4", "gridExtra", "lubridate", "raster","akima","rnaturalearth","rnaturalearthdata","sf","terra","ggquiver","scales")) </h4></center>

# 3. Data Access

[Go back to the "Table of contents"](#Table-of-contents)


From the [WEKEO viewer](https://www.wekeo.eu/data), you can explore all the products available with many filters to select the region you are interested in, the parameters you want to study, etc.

WEkEO is a European platform providing access to environmental data, developed by the European Union in partnership with agencies like ESA (European Space Agency), EUMETSAT, and Mercator Ocean. It provides access to a wide range of Earth observation data.

For this training, we will use the following data:

| Parameter | Value |
| :---: | :---|
| **Variables used** | So2 (Sulfur dioxide) |
| **Product identifier** |[EEO:ESA:DAT:SENTINEL-5P](https://wekeo.copernicus.eu/data?view=dataset&dataset=EO:ESA:DAT:SENTINEL-5P) |
| **Time Period** |on 2025-06-02 |

| Parameter | Value |
| :---: | :---|
| **Variables used** | Image |
| **Product identifier** |[EO:ESA:DAT:SENTINEL-2](https://wekeo.copernicus.eu/data?view=dataset&dataset=EO:ESA:DAT:SENTINEL-2) |
| **Time Period** |on 2025-06-02 & 2025-06-04 |

| Parameter | Value |
| :---: | :---|
| **Variables used** | u, v (Wind) |
| **Product identifier** |[EO:ECMWF:DAT:REANALYSIS_ERA5_PRESSURE_LEVELS](https://wekeo.copernicus.eu/data?view=dataset&dataset=EO:ECMWF:DAT:REANALYSIS_ERA5_PRESSURE_LEVELS) |
| **Time Period** |from 2025-06-02 to 2025-06-04|
| **level** |800 hPa & 1000 hPa|


**if you downloaded the notebook via the Atlas link, then you do not need to download the data; otherwise, here are the cells you need to run to download it.**

- query_sentinel_5P='{"dataset_id":"EO:ESA:DAT:SENTINEL-5P","bbox": [14.8,37.6,15.2,37.9],"productType": "L2__SO2___",
  "instrumentShortName": "TROPOMI","startdate": "2025-06-02T00:00:00.000Z","enddate": "2025-06-02T23:59:59.999Z"}'

- query_sentinel_2='{"dataset_id":"EO:ESA:DAT:SENTINEL-2","bbox": [14.8,37.6,15.2,37.9],"processingLevel": "S2MSI2A",
                    "platformSerialIdentifier": "S2C","startdate": "2025-06-02T00:00:00.000Z","enddate": "2025-06-02T10:00:59.999Z"}'

- query_sentinel_ERA5='{"dataset_id":"EO:ECMWF:DAT:REANALYSIS_ERA5_PRESSURE_LEVELS","product_type":["reanalysis"],
                      "variable":["u_component_of_wind","v_component_of_wind"],"year":["2025"],"month":["06"],"day":["02","03","04"],
                      "time":["00:00","01:00","02:00","03:00","04:00","05:00","06:00","07:00","08:00","09:00","10:00","11:00","12:00",
                              "13:00","14:00","15:00","16:00","17:00","18:00","19:00","20:00","21:00","22:00","23:00"],
                      "pressure_level":["800","1000"],"bbox":[10,35,20,40],"data_format":"netcdf","download_format":"unarchived"}'

__Optional:__ Go to the `Data store` and try to download this data selection or use HDA library (see [how to use it](https://help.wekeo.eu/en/articles/7035318-how-to-use-the-hdar-package-for-accessing-the-wekeo-hda-api-in-r)). Note: You'll need to have your own WEKEO credentials -- username and password. Creating an account is free of charge and available [here](https://www.wekeo.eu/register).

# 4. Input for Sentinel 5P
[Go back to the "Table of contents"](#Table-of-contents)


We will begin with Sentinel 5P data. Sentinel‑5P is a satellite of the Copernicus program dedicated to monitoring atmospheric composition. It provides daily high-resolution data on several trace gases, including sulfur dioxide (SO₂). The SO₂ variable is particularly valuable for monitoring volcanic eruptions, industrial emissions, and air pollution, allowing the mapping of their spatial distribution and assessing their impact on air quality and climate.

First we set the path for the NetCDF file:

In [ ]:
# Set the path for the NetCDF files
file <- "data/S5P_OFFL_L2_SO2.nc"

Then we load the NetCDF file with function nc_open:

In [ ]:
# Load NetCDF File
data <- nc_open(file)

And we load all the variables that we want, here: latitude, longitude, time and So2.

In [ ]:
# Access Data
latitude <- ncvar_get(data, "PRODUCT/latitude")  # Name of the latitude variable
longitude <- ncvar_get(data, "PRODUCT/longitude")  # Name of the longitude variable
time <- ncvar_get(data, "PRODUCT/time")  # Name of the time variable

so2 <- ncvar_get(data, "PRODUCT/sulfurdioxide_total_vertical_column") # Name of the SO2 variable


Here, we select some functions to visualize the characteristics of the SO₂ or time variables

In [ ]:
# visualisation of variable
print(dim(so2)) # know the dimension of the variable
print (so2[50,500]) # know the value depending on longitude and latitude

In [ ]:
# visualise time variable
time

And now, we convert the time variable to a date time variable with function POSIXct:

In [ ]:
# Variable time is the number of seconds since 2010-01-01 00:00:00, we should convert:
time_values <- as.POSIXct(time, origin = "2010-01-01", tz = "GMT")
time_values

In [ ]:
# for save memory
nc_close(data)

# 5. Sentinel 5P image

[Go back to the "Table of contents"](#Table-of-contents)
    
## 5.1. Plot of sulfur dioxide
[Go back to the "Table of contents"](#Table-of-contents)

We have loaded the variables from the Sentinel‑5P dataset. Now we are going to plot a map to visualize the data. First, we define the figure size, then we create a dataframe, perform a linear interpolation, and finally use the ggplot function to create the map.

In [ ]:
# Determine figure size
options(repr.plot.width =12, repr.plot.height = 6) 

In [ ]:
# we create a dataframe
df <- data.frame(
  Longitude = as.vector(longitude),
  Latitude = as.vector(latitude),
  so2 = as.vector(so2)
)

In [ ]:
# visualisation of the dataframe
df[1:10,]

We filter the data to create a smaller dataframe containing only the region of interest:

In [ ]:
# To select only the area of interest and obtain a smaller data frame
df <- df %>% dplyr::filter(Longitude >= 10 & Longitude <= 20, Latitude >= 35 & Latitude <= 40)


We then perform an interpolation so that we can use the dataframe with ggplot.

In [ ]:
# Interpolate onto a regular grid
df_interpolate <- with(df, interp(x = Longitude, y = Latitude, z = so2,
                                     xo = seq(10, 20, length = 200),
                                     yo = seq(35, 40, length = 200),
                                     linear = TRUE, extrap = FALSE))

In [ ]:
# Convert the result into a data frame
df_interpolate <- data.frame(expand.grid(x = df_interpolate$x, y = df_interpolate$y), z = as.vector(df_interpolate$z))
df_interpolate[1:10,]

And now we create the figure:

In [ ]:
# for creating the directory
dir.create("./figures/", recursive = TRUE, showWarnings = FALSE)

# Plotting
p <- ggplot() +

# Add SO2 data 
geom_tile(data = df_interpolate, aes(x = x, y = y, fill = z)) +

# Color scale options limits = c()
scale_fill_gradientn(colors = brewer.pal(9, "OrRd"),  name="so2[mol.m-2]",limits = c(), na.value = "grey") +

# General aesthetic options
theme_light() +

# Title and subtitle
labs(title = paste("Concentration de SO₂ Sentinel-5P - ",time_values,sep=''),
     x = "Longitude",
     y = "Latitude")+

# Legend position and aesthetic options
theme_minimal()+

theme(
    axis.text.x = element_text(size = 20),  # Increase the size of the x-axis values
    axis.text.y = element_text(size = 20),  # Increase the size of the y-axis values
    plot.title = element_text(hjust = 0.5, size = 20),  # Center and increase the size of the title
    axis.title.x = element_text(size = 20),  # Increase the size of the x-axis title
    axis.title.y = element_text(size = 20),  # Increase the size of the y-axis title
    legend.text = element_text(size = 20),    # Increase the size of the legend text
    legend.title = element_text(size = 20)     # Increase the size of the legend title
)   
#p
# to save the figure
ggsave("./figures/basic_figure.png", plot = p, width = 12, height = 6, dpi = 300)

IRdisplay::display_png(file = "./figures/basic_figure.png", width = 800, height = 600)



Here is the SO₂ map for June 2, 2025. A strong concentration of sulfur dioxide is clearly visible, with a plume extending around the source, which corresponds to the smoke plume emitted during the volcanic eruption.

## 5.2. Sulfur dioxide map with country borders
[Go back to the "Table of contents"](#Table-of-contents)

We reproduce the same plot including country borders to better visualize the geographic location.

In [ ]:
# add a background map
world <- ne_countries(scale = "medium", returnclass = "sf")


In [ ]:
# Plotting
p <- ggplot() +

# Add SO2 data 
geom_tile(data = df_interpolate, aes(x = x, y = y, fill = z)) +

# Add background map 
geom_sf(data = world, fill = NA, color = "black") +

# Color scale options limits = c()
scale_fill_gradientn(colors = brewer.pal(9, "OrRd"),  name="so2[mol.m-2]",limits = c(), na.value = "grey") +

# General aesthetic options
theme_light() +

# Title and subtitle
labs(title = paste("Concentration de SO₂ Sentinel-5P - ",time_values,sep=''),
     x = "Longitude",
     y = "Latitude")+

# Legend position and aesthetic options
theme_minimal()+

theme(
    axis.text.x = element_text(size = 20),  # Increase the size of the x-axis values
    axis.text.y = element_text(size = 20),  # Increase the size of the y-axis values
    plot.title = element_text(hjust = 0.5, size = 20),  # Center and increase the size of the title
    axis.title.x = element_text(size = 20),  # Increase the size of the x-axis title
    axis.title.y = element_text(size = 20),  # Increase the size of the y-axis title
    legend.text = element_text(size = 20),    # Increase the size of the legend text
    legend.title = element_text(size = 20)     # Increase the size of the legend title
)    +
coord_sf(xlim = c(10, 20), ylim = c(35, 40), expand = FALSE) 

# to save the figure
ggsave("./figures/basic_figure_with_borders.png", plot = p, width = 12, height = 6, dpi = 300)

IRdisplay::display_png(file = "./figures/basic_figure_with_borders.png", width = 800, height = 600)

A clear increase in SO₂ concentration is observed over Mount Etna on the day of the eruption, June 2nd. The volcanic plume and its direction can also be clearly seen.

# 6. Use of Sentinel 2 data
[Go back to the "Table of contents"](#Table-of-contents)

Now we will use satellite image data from Sentinel 2.

Sentinel‑2 is a satellite mission from the European Copernicus program, developed by the European Space Agency (ESA) to provide high-resolution optical imagery of the Earth’s land surfaces and coastal zones. It carries the MSI (MultiSpectral Instrument), which captures 13 spectral bands ranging from visible to shortwave infrared. Sentinel‑2 provides critical data for vegetation monitoring, land use mapping, disaster management, water quality assessment, and natural hazard monitoring. With its high spatial resolution and frequent revisit time, Sentinel‑2 plays a key role in observing and understanding local and regional environmental changes.

We will use Sentinel-2 data to assess the eruption. The selected dates are : June 2 and 4.

## 6.1 RGB Image During the Eruption
[Go back to the "Table of contents"](#Table-of-contents)

An RGB image is a digital image composed of three color channels: Red, Green, and Blue. By combining these channels, it represents the natural colors of a scene as perceived by the human eye.

In [ ]:
# Load Sentinel-2 bands from the June 2, 2025 eruption dataset
b2  <- rast("./data/S2_During_Eruption/S2_20250602_B02.jp2")   # Blue band (B02)
b3  <- rast("./data/S2_During_Eruption/S2_20250602_B03.jp2")   # Green band (B03)
b4  <- rast("./data/S2_During_Eruption/S2_20250602_B04.jp2")   # Red band (B04)
b8  <- rast("./data/S2_During_Eruption/S2_20250602_B08.jp2")   # Near-Infrared (NIR) band (B8A)
b11 <- rast("./data/S2_During_Eruption/S2_20250602_B11.jp2")   # Shortwave Infrared 1 (SWIR1)
b12 <- rast("./data/S2_During_Eruption/S2_20250602_B12.jp2")   # Shortwave Infrared 2 (SWIR2)
SCL <- rast("./data/S2_During_Eruption/S2_20250602_SCL.jp2")  # Scene Classification Layer (SCL)

We first define a set of coordinates to create a polygon representing an area of interest. This polygon is converted into a spatial object with the WGS84 coordinate system, then reprojected to match the bands. Finally, the polygon is used for cropping or masking raster data.

In [ ]:
# Define a matrix of coordinates (longitude, latitude) for the polygon
coords <- matrix(c(
  14.80, 37.60,
  15.20, 37.60,
  15.20, 37.90,
  14.80, 37.90,
  14.80, 37.60
), ncol = 2, byrow = TRUE)

# Create a polygon from the coordinates
poly_latlon <- st_polygon(list(coords))

# Convert the polygon into an 'sf' spatial object with WGS84 coordinate reference system
poly_sf <- st_sfc(poly_latlon, crs = 4326)  # WGS84

# Transform the polygon into the same CRS as the raster (UTM)
poly_utm <- st_transform(poly_sf, crs = crs(b8))

# Convert the 'sf' polygon into a terra vector object
poly_vect <- vect(poly_utm)

In [ ]:
# Crop each Sentinel-2 band to the area defined by the polygon
b2  <- crop(b2,  poly_vect)  # Blue band
b3  <- crop(b3,  poly_vect)  # Green band
b4  <- crop(b4,  poly_vect)  # Red band
b8  <- crop(b8,  poly_vect)  # Near-Infrared (NIR) band
b11 <- crop(b11, poly_vect)  # Shortwave Infrared 1 (SWIR1)
b12 <- crop(b12, poly_vect)  # Shortwave Infrared 2 (SWIR2)
SCL <- crop(SCL, poly_vect)  # Scene Classification Layer (SCL)

In [ ]:
# Combine Red, Green, and Blue bands to create a true-color RGB composite
rgb <- c(b4, b3, b2)

# Assign meaningful names to the layers
names(rgb) <- c("R", "G", "B")

# Perform contrast stretching to enhance visualization
rgb_stretch_during <- stretch(rgb, minq = 0.02, maxq = 0.98)

# Plot the RGB image
plotRGB(rgb_stretch_during,
        r = 1, g = 2, b = 3)
title(main = "Sentinel-2 RGB — Etna Eruption (2025)", cex.main = 1.5, col.main = "white", font.main = 2)


This Sentinel-2 RGB image clearly highlights the smoke plume and the areas affected by the current and previous eruptions.

## 6.2 Calculation of NBR during eruption
[Go back to the "Table of contents"](#Table-of-contents)

The Normalized Burn Ratio (NBR) is a spectral index used to detect burned and disturbed areas on the Earth’s surface. It is computed from the near-infrared (NIR) and short-wave infrared (SWIR) bands, which are very sensitive to changes in vegetation and soil moisture. NBR is widely used to assess fire severity and to identify surfaces such as ash, bare soil, or recently affected areas.

In [ ]:
# Calculate the Normalized Burn Ratio (NBR) using bands 8 (NIR) and 11 (SWIR)
nbr <- (b8 - b11) / (b8 + b11)

Then, we create a cloud mask:

In [ ]:
# Create a cloud mask using the Scene Classification Layer (SCL)
# Pixels with values 3, 8, 9, 10, 11 are clouds, cloud shadows, or snow
cloudmask <- SCL %in% c(3, 8, 9, 10, 11)

In [ ]:
# Apply the cloud mask to remove cloudy pixels from the NBR raster
nbr_clear <- mask(nbr, cloudmask, maskvalues=TRUE)

# Crop the NBR raster to the area of interest defined by poly_vect
nbr_crop_during <- crop(nbr_clear, poly_vect)

In [ ]:
# Plot the resulting NBR raster
plot(nbr_crop_during, main="NBR")

The NBR image shows white areas corresponding to clouds, which have been removed using the cloud mask. Some negative values are observed and correspond to volcanic ash or bare volcanic surfaces. The majority of the NBR values are below 0.2, indicating sparse vegetation or mostly bare ground, while only a few pixels exceed 0.2, showing that areas with healthy vegetation are very limited.

## 6.3. Comparison between during and after the eruption
[Go back to the "Table of contents"](#Table-of-contents)

### 6.3.1. RGB images
[Go back to the "Table of contents"](#Table-of-contents)

Up to now, we have analyzed the main characteristics of the eruption. We will now compare the different images in order to visualize and assess the impact of the eruption on the surrounding areas. This comparison will help identify the affected zones and evaluate the extent of the changes caused by the event.

In [ ]:
# Load Sentinel-2 bands from the June 4, 2025 eruption dataset
b2  <- rast("./data/S2_After_Eruption/S2_20250604_B02.jp2")   # Blue band (B02)
b3  <- rast("./data/S2_After_Eruption/S2_20250604_B03.jp2")   # Green band (B03)
b4  <- rast("./data/S2_After_Eruption/S2_20250604_B04.jp2")  # Red band (B04)
b8  <-  rast("./data/S2_After_Eruption/S2_20250604_B08.jp2")   # Near-Infrared (NIR) band (B8A)
b11 <- rast("./data/S2_After_Eruption/S2_20250604_B11.jp2")   # Shortwave Infrared 1 (SWIR1)
SCL <- rast("./data/S2_After_Eruption/S2_20250604_SCL.jp2")  # Scene Classification Layer (SCL)

# Crop each Sentinel-2 band to the area defined by the polygon
b2  <- crop(b2,  poly_vect)  # Blue band
b3  <- crop(b3,  poly_vect)  # Green band
b4  <- crop(b4,  poly_vect)  # Red band
b8  <- crop(b8,  poly_vect)  # Near-Infrared (NIR) band
b11 <- crop(b11, poly_vect)  # Shortwave Infrared 1 (SWIR1)
b12 <- crop(b12, poly_vect)  # Shortwave Infrared 2 (SWIR2)
SCL <- crop(SCL, poly_vect)  # Scene Classification Layer (SCL)

In [ ]:
# Combine Red, Green, and Blue bands to create a true-color RGB composite
rgb <- c(b4, b3, b2)

# Assign meaningful names to the layers
names(rgb) <- c("R", "G", "B")

# Perform contrast stretching to enhance visualization
rgb_stretch_after <- stretch(rgb, minq = 0.02, maxq = 0.98)

In [ ]:
# Set the plotting area to 1 row and 2 columns
par(mfrow = c(1, 2))

# Left image
plotRGB(rgb_stretch_during , r = 1, g = 2, b = 3)
title(main = "During eruption", cex.main = 1.5, col.main = "red", font.main = 2)

# Right image
plotRGB(rgb_stretch_after , r = 1, g = 2, b = 3)
title(main = "After eruption", cex.main = 1.5, col.main = "red", font.main = 2)

# Reset layout to default (important after)
par(mfrow = c(1, 1))

During the eruption, smoke is clearly visible in the RGB image, whereas it is almost absent two days later. Only minor changes are observed in the vegetation, suggesting that the lava mainly flowed over areas already burned by previous eruptions. The green, vegetated areas remain intact.

### 6.3.2. NBR images
[Go back to the "Table of contents"](#Table-of-contents)

In [ ]:
# Calculate NBR
nbr <- (b8 - b11) / (b8 + b11)

# Create a cloud mask 
cloudmask <- SCL %in% c(3, 8, 9, 10, 11)

# Apply the cloud mask to remove cloudy pixels
nbr_clear <- mask(nbr, cloudmask, maskvalues=TRUE)

# Crop the NBR raster 
nbr_crop_after <- crop(nbr_clear, poly_vect)



In [ ]:
# Determine figure size
options(repr.plot.width =12, repr.plot.height = 6) 
# Set the plotting area to 1 row and 2 columns
par(mfrow = c(1, 2))
breaks <- seq(-0.6, 0.3, length.out = 6)
# Left image
plot(nbr_crop_during,breaks=breaks, legend = TRUE)
title(main = "During eruption", cex.main = 1.5, col.main = "red", font.main = 2, line = 3)

# Right image
plot(nbr_crop_after,breaks=breaks, legend = FALSE)
title(main = "After eruption", cex.main = 1.5, col.main = "red", font.main = 2, line = 3)
# Reset layout to default (important after)
par(mfrow = c(1, 1))


The lava is clearly visible in light pink on the image captured during the eruption, whereas this coloration is no longer present two days later. Examining the direction and extent of the lava, no significant differences are observed between the two images. This suggests that the lava affected only areas that were already usually burned or devoid of vegetation.

# 7. Visualisation of wind 
[Go back to the "Table of contents"](#Table-of-contents)

We will now visualize the wind using ERA5 data to understand the direction of the smoke. This is important because the ash and smoke released during the eruption can pose hazards to local populations or aviation.

In [ ]:
# Set the path for the NetCDF files
file <- "data/wind.nc"

# Load NetCDF File
data <- nc_open(file)

# Access Data wind
latitude <- ncvar_get(data, "latitude")  # Name of the latitude variable
longitude <- ncvar_get(data, "longitude")  # Name of the longitude variable
pressure_level<- ncvar_get(data, "pressure_level")  # Name of the pressure level variable

time <- ncvar_get(data, "valid_time")  # Name of the time variable
time_values <- as.POSIXct(time, origin = "1970-01-01", tz = "GMT")
time_values[1:10]

u_wind <- ncvar_get(data, "u")       # eastward wind
v_wind <- ncvar_get(data, "v")       # northward wind


In [ ]:
# we convert wind in norm
wind <- sqrt(u_wind^2 + v_wind^2)

In [ ]:
# we create a dataframe only for the second pressure level
index_time=6
u=u_wind[,,2,index_time]
v=v_wind[,,2,index_time]
norm=wind[,,2,index_time]

df <- data.frame(expand.grid(longitude, latitude,time_values), 
                 u = as.vector(u),v = as.vector(v),norm=as.vector(norm))

names(df) <- c("Longitude", "Latitude","time","u", "v","wind")

# visualisation of the dataframe
df[1:10,]

We create a second dataframe specifically for plotting the arrows on the map. It's based on the original dataframe, but we reduce the number of points to make the arrow visualization clearer and more readable.

In [ ]:
step <- 0.75

df_sparse <- df %>%
  mutate(
    Lon_grid = round(Longitude / step) * step,
    Lat_grid = round(Latitude / step) * step
  ) %>%
  group_by(Lon_grid, Lat_grid) %>%
  slice(1) %>%  # Keep only one point per grid cell
  ungroup()

# visualisation of the dataframe
df_sparse[1:10,]

In [ ]:
# Download a world map at medium resolution as an 'sf' (simple features) object
world <- ne_countries(scale = "medium", returnclass = "sf")

# Ensure the object is treated as an 'sf' object (spatial features) for compatibility with ggplot2
world <- st_as_sf(world)

And finally, we create the map of wind:

In [ ]:
# Determine figure size
options(repr.plot.width =12, repr.plot.height = 8) 
# Plotting
p <- ggplot() +

# Add wind data 
geom_tile(data = df, aes(x = Longitude, y = Latitude, fill = wind)) +

# vector display
geom_quiver(data = df_sparse, aes(x = Longitude, y = Latitude, u = u, v = v), color = "black", size = 0.7, vecsize= 2) +

# Color scale options limits
scale_fill_viridis_c(name = "Norm (m/s)",limits =c(0,15), oob = scales::squish) +

# add contour of countries
geom_sf(data = world, fill = NA, color = "black") +
coord_sf(xlim = c(min(df$Longitude), max(df$Longitude)),ylim = c(min(df$Latitude), max(df$Latitude))) +
# Title and subtitle
labs(title = paste(" wind speed [m/s] at 800hPa (around 2000m) - ",time_values[index_time],sep=''),
     x = "Longitude",
     y = "Latitude")+

# Legend position and aesthetic options
theme_minimal()+
theme_light() +

theme(
    axis.text.x = element_text(size = 20),  # Increase the size of the x-axis values
    axis.text.y = element_text(size = 20),  # Increase the size of the y-axis values
    plot.title = element_text(hjust = 0.5, size = 20),  # Center and increase the size of the title
    axis.title.x = element_text(size = 20),  # Increase the size of the x-axis title
    axis.title.y = element_text(size = 20),  # Increase the size of the y-axis title
    legend.text = element_text(size = 20),    # Increase the size of the legend text
    legend.title = element_text(size = 20)     # Increase the size of the legend title
)   +
  guides(fill = guide_colorbar(barwidth = 2, barheight = 15))  # Adjust the size of the colorbar
 
p
# to save the figure
ggsave("./figures/basic_figure_wind.png", plot = p, width = 12, height = 12, dpi = 300)



Here is the wind speed at 800 hPa, which helps to understand the direction of smoke dispersion. Multiple images can be produced at different times to follow the evolution of the smoke. Additionally, the same analysis can be performed at different altitudes to study how the smoke moves through different layers of the atmosphere. If you want, you can try!

# 8. Conclusion
[Go back to the "Table of contents"](#Table-of-contents)

<div class="alert alert-block alert-success">
<b>Congratulations!</b> You have successfully completed the introductory-intermediate tutorial on using WEkEO products to evaluate the Etna eruption. Throughout this tutorial, we have explained the basic tools necessary to access and visualize observation and satellite data, generate different types of plots.
<br><br>

In this tutorial, you acquired all the information you need to:
 

* Access NetCDF datasets and Multispectral images.

* Navigate through the different variables, dimensions, and attributes of a NetCDF file.

* Plot maps of any variable.

* Modify maps to include additional information.

 
We sincerely hope that you have enjoyed the tutorial and found useful information in it. Please keep in mind that the tutorial has a progressive difficulty, moving quickly from basic elements to intermediate levels. Our intention is for all users to find useful information tailored to their level.
 
We understand that, for a user without prior knowledge, fully understanding all the procedures in the tutorial may be a challenge that requires some effort. However, we encourage everyone to take on the challenge as this is just the beginning of a journey towards a new understanding of the earth and its ecosystems.

Credit: This notebook was developed by [NOVELTIS](https://www.noveltis.fr/) under Contract No. 25335-COP-CAPACITY-DEV-8000 with Mercator Ocean. 
</div>

 